# 14 · Gruppi, aggiornamenti e job: cosa c'è davvero e come si presenta

Questo notebook è **onesto sullo stato**. Gli **aggiornamenti software**, i **gruppi** e i **job** di ADR sono in **preview** (documentazione Microsoft Learn del 5-12 novembre 2026).
Qui non si simula niente: ci sono lo **stato** della preview e i **comandi veri** della CLI. La parte che si esegue su Azure è nel notebook 15 (gruppi e software revision).

| Disponibile in preview | **Non** disponibile in preview |
|---|---|
| Gruppi definiti da una **query** sui device (membership in cache, refresh) | **Ondate, canary, soglia di fallimento, rollback automatico** |
| Aggiornamento = file + **import manifest** (stesso formato di Device Update) | **Retry** dei soli device falliti (si crea un nuovo job) |
| Job `SoftwareUpdate` (su un gruppo) e `OnboardingUpdate` (sul namespace) | Device **Azure IoT Operations** (solo device collegati a IoT Hub) |
| Run pianificate, annullabili, con risultato **per device** | Un Hub che usa già **Device Update for IoT Hub** |
| CLI `az iot adr ns job ...` e portale | Import manifest creato dal portale; client per **PowerShell** |

Il device riceve gli aggiornamenti con il **software updates client** degli *unified Azure IoT SDKs (preview)*, pensato per Linux con più di 32 MB di memoria.
Per produzione Microsoft indica ancora **Device Update for IoT Hub** (GA).

**Attenzione:** canary, soglia di fallimento e rollback **non sono funzioni dei job ADR**.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## I comandi veri
Quelli che userebbe chi fa operation, dalla documentazione della CLI (`azure-iot` extension, gruppo **in preview**).
**Qui non vengono eseguiti**: il job vero è nei notebook 14a (servizio), 14b (import), 14c e 14d (job, run e device).
Le celle sotto stampano solo l'**aiuto** dei comandi, che è in sola lettura.

In [ ]:
@'
# 1. importare l'update (guida: "Import a software update" su Microsoft Learn), poi un gruppo (leggi: az iot adr ns group create -h)
# 2. il job: SoftwareUpdate su un gruppo, oppure OnboardingUpdate sul namespace (senza gruppo)
az iot adr ns job create -n aggiorna-camion --ns $adrName -g $rg --type SoftwareUpdate --target-group-name trucks `
   --update-id-provider Contoso --update-id-name truck-fw --update-id-version 2.0.0
# 3. la run: subito, oppure a un orario UTC
az iot adr ns job schedule -n aggiorna-camion --ns $adrName -g $rg [--scheduled-time 2026-12-01T08:00:00Z] [--run-name r1]
# 4. osservare
az iot adr ns job run summary ...        # contatori
az iot adr ns job run results ...        # risultato per device
# 5. chiudere
az iot adr ns job run cancel ...
'@

# 1. importare l'update (guida: "Import a software update" su Microsoft Learn), poi un gruppo (leggi: az iot adr ns group create -h)
# 2. il job: SoftwareUpdate su un gruppo, oppure OnboardingUpdate sul namespace (senza gruppo)
az iot adr ns job create -n aggiorna-camion --ns $adrName -g $rg --type SoftwareUpdate --target-group-name trucks `
   --update-id-provider Demo --update-id-name truck-fw --update-id-version 2.0.0
# 3. la run: subito, oppure a un orario UTC
az iot adr ns job schedule -n aggiorna-camion --ns $adrName -g $rg [--scheduled-time 2026-12-01T08:00:00Z] [--run-name r1]
# 4. osservare
az iot adr ns job run summary ...        # contatori
az iot adr ns job run results ...        # risultato per device
# 5. chiudere
az iot adr ns job run cancel ...

In [ ]:
az iot adr ns job create -h
az iot adr ns job run -h
az iot adr ns group create -h
az iot adr ns su -h

Command

    az iot adr ns job create : Create a job in a Device Registry namespace.

        PUT is a long-running operation. SoftwareUpdate jobs require a target group.

        OnboardingUpdate jobs target all compatible onboarding devices and do not

        accept a target group. Job definition fields are immutable after creation.

        The target group is specified by --target-group-name and must live in the

... (139 outputs left out)

Subgroups:

    device-class    : View and delete Software Updates device classes.

    instance        : Manage the Update Instance that powers Software Updates.

    software-update : Import and manage Software Updates content.

## Cosa abbiamo visto
- Il **gruppo** è una query con membership **in cache**; il **job** è la definizione, la **run** l'esecuzione, e il risultato è **per device**.
- La run **resta attiva** e serve i ritardatari (i device offline che tornano), finché non la chiudi. I **nuovi membri** del gruppo arrivano solo dopo un refresh della membership (una volta all'ora).
- L'integrità è verificata dal device con gli **hash** del manifest.
- **Non ci sono** canary, ondate, soglie, rollback né retry: se servono, vanno costruiti sopra (o si usa Device Update for IoT Hub, GA, che ha altre caratteristiche).

**Prossimo notebook: 14a (crea Azure Update Service), poi 14b, 14c, 14d; il 15 per i gruppi e la software revision.**